# Breast Cancer Classification — Local Training

Trains on **BreaKHis** dataset using the modular pipeline.
Configured for your local paths. Run cells sequentially.

In [ ]:
# 1. Imports & Config
import sys, os
PROJECT_DIR = r"C:\Users\hp\Downloads\breast_cancer_project"
sys.path.insert(0, PROJECT_DIR)
os.chdir(PROJECT_DIR)

from src.config import Config
from src.models.model_factory import create_model
from src.datasets.dataloader import DataLoaderFactory
from src.training.trainer import Trainer
from src.evaluation.metrics import ClassificationMetrics
from src.evaluation.visualizer import Visualizer
from src.evaluation.gradcam import GradCAMView
from src.utils.logger import Logger

import torch
import numpy as np
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# 2. Load Config & Model
config = Config(r"C:\Users\hp\Downloads\breast_cancer_project\config\config.yaml")
config.cfg.data.breakhis.path = r"C:\Users\hp\Downloads\breakhis\BreaKHis_v1\BreaKHis_v1\histology_slides\breast"
config.cfg.data.inbreast.use = False
config.cfg.data.mias.use = False
config.cfg.model.name = "efficientnet_b3"
config.cfg.training.epochs = 100
config.cfg.training.batch_size = 32
config.cfg.training.learning_rate = 0.001
config.cfg.training.mixed_precision = torch.cuda.is_available()

model = create_model(config)
print(f"Model: {config.model.name}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Device: {config.device}")

In [ ]:
# 3. Create DataLoaders
loader_factory = DataLoaderFactory(config)
train_loader, val_loader, test_loader = loader_factory.get_dataloaders()
print(f"Train: {len(train_loader.dataset)}")
print(f"Val:   {len(val_loader.dataset)}")
print(f"Test:  {len(test_loader.dataset)}")

In [ ]:
# 4. Train
trainer = Trainer(model, config)
best_acc = trainer.fit(train_loader, val_loader)
print(f"Best val accuracy: {best_acc:.4f}")

In [ ]:
# 5. Evaluate on Test Set
model.eval()
all_outputs, all_labels = [], []
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(config.device)
        labels = labels.to(config.device)
        outputs = torch.softmax(model(images), dim=1)
        all_outputs.append(outputs.cpu().numpy())
        all_labels.append(labels.cpu().numpy())

all_outputs = np.concatenate(all_outputs)
all_labels = np.concatenate(all_labels)
all_preds = all_outputs.argmax(axis=1)

metrics = ClassificationMetrics(num_classes=2)
report = metrics.compute(all_labels, all_preds, all_outputs)

print(f"Accuracy:    {report['accuracy']:.4f}")
print(f"Precision:   {report['precision']:.4f}")
print(f"Recall:      {report['recall']:.4f}")
print(f"F1:          {report['f1_score']:.4f}")
print(f"Sensitivity: {report['sensitivity']:.4f}")
print(f"Specificity: {report['specificity']:.4f}")
print(f"AUC-ROC:     {report['auc_roc']:.4f}")

In [ ]:
# 6. Plot Results
viz = Visualizer(config.output.plot_dir)
viz.plot_confusion_matrix(np.array(report['confusion_matrix']), ['benign', 'malignant'])
viz.plot_roc_curve(all_labels, all_outputs, ['benign', 'malignant'])
viz.plot_training_history(
    trainer.train_losses, trainer.val_losses,
    trainer.train_accs, trainer.val_accs
)
viz.plot_metrics_comparison(report)
viz.save_metrics_report(report)
print(f"Plots saved to {config.output.plot_dir}")

In [ ]:
# 7. Save Final Model
import os, torch
model_path = os.path.join(config.output.model_dir, f"{config.model.name}_final.pth")
torch.save({
    'model_state_dict': model.state_dict(),
    'config': config.cfg,
    'test_accuracy': report['accuracy'],
    'test_report': report
}, model_path)
print(f"Model saved: {model_path}")

In [ ]:
# 8. Grad-CAM Visualization (on 5 test samples per class)
from PIL import Image
from src.preprocessing.augmentations import pil_to_numpy, get_val_transforms

gradcam = GradCAMView(model, target_layers=[], device=config.device)
transforms = get_val_transforms(config)

class_samples = {0: [], 1: []}
for idx in range(len(test_loader.dataset)):
    label = test_loader.dataset.labels[idx]
    if len(class_samples[label]) < 5:
        class_samples[label].append(idx)

for cls, indices in class_samples.items():
    name = ['benign', 'malignant'][cls]
    for i, idx in enumerate(indices):
        img = Image.open(test_loader.dataset.file_paths[idx]).convert('RGB')
        img_np = pil_to_numpy(img)
        tensor = transforms(image=img_np)['image'].unsqueeze(0).to(config.device)
        heatmaps = gradcam.generate_heatmap(tensor, class_idx=cls)
        for layer, hm in heatmaps.items():
            path = os.path.join(config.output.gradcam_dir, f'{name}_{i}_{layer}.png')
            gradcam.save_heatmap(img_np, hm, path)
print(f"Grad-CAM images saved to {config.output.gradcam_dir}")

In [ ]:
# 9. Display Sample Plots
from IPython.display import Image as IPImage, display
import glob
for png in glob.glob(os.path.join(config.output.plot_dir, '*.png')):
    print(os.path.basename(png))
    display(IPImage(png))